In [55]:
from __future__ import annotations
import pandas as pd, numpy as np
from pathlib import Path
from typing import Optional, Literal
from pandas.api.types import is_numeric_dtype, is_bool_dtype
import json
import warnings
warnings.filterwarnings("ignore")

# Evidently
from evidently import Report
from evidently.presets import DataDriftPreset
from evidently.metrics import ValueDrift, DriftedColumnsCount
from evidently import DataDefinition, Dataset

In [ ]:
plant_files = {
    "planta1": Path("../df_procesados/df_planta_1.csv"),
    "planta2": Path("../df_procesados/df_planta_2.csv"),
    "planta3": Path("../df_procesados/df_planta_3.csv"),
}
flag_files = {
    "planta1": Path("../df_procesados/flags_p1.csv"),
    "planta2": Path("../df_procesados/flags_p2.csv"),
    "planta3": Path("../df_procesados/flags_p3.csv")
}
ROOT_OUT = Path("../reportes_Drift")  # carpeta raíz; dentro se crea subcarpeta por planta
ROOT_OUT.mkdir(parents=True, exist_ok=True)

# Ventana actual (reducir falsos positivos)
CURRENT_WINDOW: str = "24H"  # 24 horas

# Baseline (determinístico)
BASELINE_STRATEGY: Literal["golden","decay","seasonal"] = ""
DECAY_HALF_LIFE_HOURS: int = 24*7
DECAY_WEIGHT_MASS: float = 0.95
GOLDEN_WIN: str = "60min"
GOLDEN_STEP: str = "10min"
GOLDEN_K: int = 60
SEASONAL_WEEKS_BACK: int = 12

# Columnas a excluir
EXCLUDE_COLUMNS = [
    'pH Ecualizador 2 (Tk 250m3)', "Conductividad DAF", "Temperatura DAF",
    'pH entrada a Ecualizador 1', "Flujo Aire Reactor 1", "OD Reactor 1"
]

# Método Evidently numérico (estricto para menos falsos positivos)
NUM_METHOD: Literal["ks"] = "ks"
#NUM_METHOD: Literal["auto","ks","wasserstein","psi","anderson","cramer","mannwhitney"] = "auto"

NUM_THRESHOLD: float = 0.01  # 1%

In [57]:
def strip_outliers(df: pd.DataFrame) -> pd.DataFrame:
    if "is_outlier" not in df.columns: return df
    m = ~(df["is_outlier"].astype(str).str.lower().isin(["1","true","t","yes","y"]))
    return df.loc[m].drop(columns=["is_outlier"])

def integrate_flags(df: pd.DataFrame, flag_path: Path | None) -> pd.DataFrame:
    if not flag_path or not flag_path.exists():
        print("[flags] No encontrados: se usa DF completo.")
        return df

    flags = pd.read_csv(flag_path, parse_dates=["date_time"])
    flags["date_time"] = pd.to_datetime(flags["date_time"]).dt.floor("min")

    df = df.copy()
    if not isinstance(df.index, pd.DatetimeIndex):
        if "date_time" in df.columns:
            df["date_time"] = pd.to_datetime(df["date_time"], errors="coerce")
            df = df.dropna(subset=["date_time"]).set_index("date_time")
        else:
            df.index = pd.to_datetime(df.index, errors="coerce")
            df = df[~df.index.isna()]

    df.index = df.index.floor("min")
    df = df.merge(flags, left_index=True, right_on="date_time", how="left").set_index("date_time")

    nd_cols = [c for c in df.columns if c.startswith("nd_")]
    for nd_col in nd_cols:
        var = nd_col.replace("nd_", "")
        if var in df.columns:
            # ✅ considerar NaN en flag como válido (no anular)
            valid = df[nd_col].astype("boolean").fillna(True)
            df.loc[~valid, var] = np.nan

    drop_cols = ["valid_for_drift", "nd_any", "nd_all"] + nd_cols
    keep = [c for c in df.columns if c not in drop_cols]
    df = df[keep]

    print(f"[flags] Integrados → {len(df)} filas (sin eliminar registros completos)")
    return df

def window_starts(index: pd.DatetimeIndex, win: pd.Timedelta, step: pd.Timedelta):
    if len(index) == 0: return []
    t, tmax, out = index.min(), index.max(), []
    while t + win <= tmax:
        out.append(t); t = t + step
    return out

In [58]:
def ref_decay_prefix_mass(df_hist: pd.DataFrame, now: pd.Timestamp,
                          half_life_hours=24*7, target_mass=0.95) -> pd.DataFrame:
    if df_hist.empty: return df_hist
    tau = pd.Timedelta(hours=half_life_hours) / np.log(2)
    w = np.exp(-(now - df_hist.index) / tau).astype(float)
    order = np.argsort(-df_hist.index.view("i8"))
    cum = np.cumsum(w.values[order]) / w.values.sum()
    cut = np.searchsorted(cum, target_mass, side="left")
    take = order[:cut+1]
    return df_hist.iloc[np.sort(take)]

def ref_golden_minuto_a_minuto(df_hist: pd.DataFrame,
                               win="60min", step="10min", k=60) -> pd.DataFrame:
    """Golden sin resample: elige K ventanas históricas más estables."""
    win_td, step_td = pd.to_timedelta(win), pd.to_timedelta(step)
    starts = window_starts(df_hist.index, win_td, step_td)
    if not starts: return df_hist.iloc[:0]
    filas = []
    for t0 in starts:
        t1 = t0 + win_td - pd.Timedelta(nanoseconds=1)
        sub = df_hist.loc[t0:t1]
        if len(sub) < 3: continue
        num = sub.select_dtypes(include="number")
        if num.shape[1] == 0: continue
        med = num.median()
        iqr = num.quantile(0.75) - num.quantile(0.25)
        rsd = (iqr / (med.abs() + 1e-12)).replace([np.inf, -np.inf], np.nan)
        score = rsd.median(skipna=True)
        filas.append((t0, t1, float(score)))
    if not filas: return df_hist.iloc[:0]
    stab = pd.DataFrame(filas, columns=["t0","t1","score"]).sort_values("score").head(k)
    return pd.concat([df_hist.loc[t0:t1] for t0, t1, _ in stab.itertuples(index=False)],
                     axis=0) if len(stab) else df_hist.iloc[:0]

def ref_seasonal(df_hist: pd.DataFrame, current_end: pd.Timestamp, weeks_back=12) -> pd.DataFrame:
    if df_hist.empty: return df_hist.iloc[:0]
    slot = current_end.dayofweek * 24 + current_end.hour
    mask = (df_hist.index.dayofweek * 24 + df_hist.index.hour) == slot
    hist = df_hist.loc[mask].loc[:current_end]
    if hist.empty: return df_hist.iloc[:0]
    return hist.loc[current_end - pd.Timedelta(weeks=weeks_back):]

In [59]:
def extract_value_drift_table(report) -> pd.DataFrame:
    d = report.as_dict() if hasattr(report, "as_dict") else json.loads(report.json())
    rows = []
    for m in d.get("metrics", []):
        if m.get("metric") == "ValueDrift":
            r = m.get("result", {}) or {}
            rows.append({
                "col": r.get("column_name") or r.get("column"),
                "drifted": r.get("drift_detected"),
                "score": r.get("drift_score"),
                "method": r.get("stattest_name") or r.get("stattest"),
                "threshold": r.get("drift_threshold") or r.get("threshold"),
            })
    return pd.DataFrame(rows)

In [ ]:
def make_report_for_plant(
    df: pd.DataFrame,
    strategy: Literal["golden","decay","seasonal"] = BASELINE_STRATEGY,
    out_prefix: str = "planta",
) -> Path:

    # Índice temporal fijo + outliers
    df = ensure_datetime_index(df, "date_time")   # ✅ convierte a DatetimeIndex
    df = strip_outliers(df)
    if df.empty: raise ValueError("Dataset vacío tras limpieza.")

    # Flags (si existen)
    flag_path = flag_files.get(out_prefix)
    df = integrate_flags(df, flag_path)

    # Split temporal (24H actuales vs histórico)
    now = df.index.max()
    cur_start = now - pd.to_timedelta(CURRENT_WINDOW)
    cur = df.loc[cur_start:now]
    hist = df.loc[:cur_start - pd.Timedelta(nanoseconds=1)]

    # Baseline
    if strategy == "decay":
        ref_global = ref_decay_prefix_mass(hist, now, DECAY_HALF_LIFE_HOURS, DECAY_WEIGHT_MASS)
    elif strategy == "seasonal":
        ref_global = ref_seasonal(hist, now, SEASONAL_WEEKS_BACK)
    else:  # golden por defecto
        ref_global = ref_golden_minuto_a_minuto(hist, GOLDEN_WIN, GOLDEN_STEP, GOLDEN_K)
    if ref_global.empty: ref_global = hist

    # Columnas comunes (menos excluidas)
    common = sorted(set(ref_global.columns).intersection(cur.columns) - set(EXCLUDE_COLUMNS))
    if not common: raise ValueError("No hay columnas comunes para comparar.")
    ref_final, cur_final = ref_global[common].copy(), cur[common].copy()

    # Tipos para Evidently
    numeric_cols, categorical_cols = [], []
    for c in common:
        r, k = ref_final[c], cur_final[c]
        if r.dropna().empty and k.dropna().empty: 
            continue
        if is_bool_dtype(r) or is_bool_dtype(k): categorical_cols.append(c)
        elif is_numeric_dtype(r) or is_numeric_dtype(k): numeric_cols.append(c)
        else: categorical_cols.append(c)
    if not numeric_cols and not categorical_cols:
        raise ValueError("Sin columnas válidas (todas NaN).")

    definition = DataDefinition(
        numerical_columns=numeric_cols or None,
        categorical_columns=categorical_cols or None
    )

    # Métricas Evidently (KS 0.01)
    preset_kwargs = {"num_method": NUM_METHOD, "num_threshold": NUM_THRESHOLD}
    metrics = [
        DataDriftPreset(**preset_kwargs),
        DriftedColumnsCount(**preset_kwargs),
        *[ValueDrift(column=c, method=NUM_METHOD, threshold=NUM_THRESHOLD)
          for c in (numeric_cols + categorical_cols)],
    ]

    report = Report(metrics=metrics)
    ds_ref = Dataset.from_pandas(ref_final.reset_index(drop=True), data_definition=definition)
    ds_cur = Dataset.from_pandas(cur_final.reset_index(drop=True), data_definition=definition)
    report.run(reference_data=ds_ref, current_data=ds_cur)

    out_html = output_dir / f"{out_prefix}_{strategy}_24H.html"
    report.save_html(str(out_html))

    t = extract_value_drift_table(report)
    print(f"[{out_prefix}] cur={len(cur)} | ref={len(ref_final)} | cols={len(t)} | drifted={int(t['drifted'].fillna(False).sum())} | {out_html.name}")
    return out_html

In [61]:
for plant, path in plant_files.items():
    df = pd.read_csv(path)
    make_report_for_plant(df, strategy=BASELINE_STRATEGY, out_prefix=plant)

[flags] Integrados → 82286 filas (sin eliminar registros completos)


ValueError: An empty column 'Flujo Parshall 01 entrada a Ecualizador 1' was provided for drift calculation in the reference dataset.